## Thorlabs Spectrometer. Data Processing Part 2

**Dataset:** `\2026_09_11_spec\p4p5Pa_fast_scan` 

**Steps:** plot figs


In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
##### import project related moduls ####
current_file = Path.cwd() # cwd = path/*.ipynb - does not work in .py files.
print(f"current_file = {current_file}")
project_root = current_file.parent.parent
print(f"project_root = {project_root}")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import utils.file_utils as fu   
from utils.plt_styler_avp import PlotStyler

%matplotlib QtAgg

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
folder_path=Path(r"C:\Andrei\DATA\VINETA_75\2026_09_11_spec\p4p5Pa_fast_scan\tables")
# folder_path = None
files = fu.list_files_in_folder(folder_path=folder_path)
list_files = []
for file in files:
    print(file.parent)
    print(file.name)
    list_files.append(file)


In [ ]:
for f in list_files:
    print(f)

In [ ]:
## load intensity data
print(list_files[0])
print(list_files[1])
spectr_A = pd.read_csv(list_files[0], sep="\t",
                       index_col=0
                       )

spectr_B = pd.read_csv(list_files[1], sep="\t",
                       index_col=0
                       )

# print(spectr_A.head(2))
# print(spectr_B.head(2))
print(spectr_A.info())



### Fig Spectrum vs pixel 

In [ ]:
%reload_ext autoreload

In [ ]:
def get_spec_data(exp_index, spec_df):
    df = spec_df.copy()
    index = exp_index - 1
    if index < 0:
        print(f"Error: index={index} < 0")
        return
    spectrum_row = df.iloc[index]
    power = spectrum_row["power"]
    intensity = spectrum_row.drop("power")
    intensity.index = intensity.index.astype(int)
    return {"x": intensity.index,
            "y": intensity.values,
            "power":power}
    

In [ ]:
spec_01A = get_spec_data(1, spectr_A)
spec_15A = get_spec_data(15, spectr_A)
spec_01B = get_spec_data(1, spectr_B)
spec_15B = get_spec_data(15, spectr_B)

In [ ]:
def make_spec_fig(figsize=(20, 5), dpi=100):

    styler = PlotStyler()
    styler.set_plt_font_style(size=20, profile='default')
    fig, ax = plt.subplots(figsize=figsize, dpi=dpi)
    styler.set_scale_steps(ax)

    ax.minorticks_on()
    ax.tick_params(which="major", length=8, width=1.5, direction="out")
    ax.tick_params(which="minor", length=4, width=1.0, direction="out")
    for spine in ax.spines.values():
            spine.set_linewidth(1.5)

    ax.set_xlabel("pixel index") 
    ax.set_ylabel("intensity [counts/s]") 
    fig.tight_layout()

    return fig, ax


In [ ]:
fig_A, ax_A = make_spec_fig()
fig_B, ax_B = make_spec_fig()

In [ ]:
### "tab:blue"   "tab:orange"   "tab:green" "tab:red"  "tab:purple"  "tab:brown"  "tab:pink"
### "tab:gray"   "tab:olive"   "tab:cyan"*
def add_line( ax, data, color="tab:blue", marker="."):
    ax.plot( data['x'], 
        data['y'], 
        color=color, 
        marker=marker, 
        linewidth=1.2, 
        label=f"power = {data['power']} kW", 
        )

    ax.legend() 
    ax.figure.canvas.draw_idle()
    #plt.tight_layout() 
    #plt.show()


In [ ]:

add_line(ax_A, spec_01A, color="tab:blue",  marker=".")
add_line(ax_A, spec_15A, color="tab:orange",  marker=".")

add_line(ax_B, spec_15B, color="tab:orange",  marker=".")
add_line(ax_B, spec_01B, color="tab:blue",  marker=".")


In [ ]:
def mark_pixel( ax, pixel, label=None, color="gray", linestyle="--", linewidth=1.2, y_pos=0.95, ):
    pixel = int(pixel)
    text  = f"pixel {pixel}"
    ax.axvline(x=pixel, color=color, linestyle=linestyle, linewidth=linewidth, alpha=0.8, )
    ax.text(
        x=pixel,
        y=y_pos,
        s=f" {text}",
        rotation=90,
        verticalalignment="top",
        horizontalalignment="right",
        color=color,
        transform=ax.get_xaxis_transform(),
    )
    ax.figure.canvas.draw_idle()


In [ ]:
mark_pixel(ax_A, 3611) ## strongest Ar line on the backside
mark_pixel(ax_A, 3262) ## strongest Ar line near antenna
mark_pixel(ax_A, 3183) ## weak Ar? line test for ratio 
mark_pixel(ax_A, 2478) ## weak Ar? line test for ratio 
mark_pixel(ax_A, 1028) ## strong Ar+ line


mark_pixel(ax_B, 3611-11) ## strongest Ar line on the backside
mark_pixel(ax_B, 3262-11) ## strongest Ar line near antenna
mark_pixel(ax_B, 3183-11) ## weak Ar? line test for ratio 
mark_pixel(ax_B, 2478-11) ## weak Ar? line test for ratio 
mark_pixel(ax_B, 1028-11) ## strong Ar+ line



In [ ]:
print(spec_01A.keys())

In [ ]:
def intensity_normalization(data: dict, pixel: int) -> dict:
    norm_data = data.copy()
    x_arr = np.asarray(data["x"])
    y_arr = np.asarray(data["y"])
    mask = x_arr == pixel
    if not np.any(mask):
        print("pixel ERROR")
        return norm_data
    ref_val = y_arr[mask]
    if ref_val == 0:
        print(f"could not normalize, intensity zerro")
        return norm_data
    norm_data["y"] = y_arr / ref_val
    return norm_data
    


In [ ]:
spec_01A3262 = intensity_normalization(spec_01A, pixel=3262)
spec_15A3262 = intensity_normalization(spec_15A, pixel=3262)

In [ ]:
fig_An, ax_An = make_spec_fig()

In [ ]:
add_line(ax_An, spec_01A3262, color="tab:blue",  marker=".")
add_line(ax_An, spec_15A3262, color="tab:orange",  marker=".")

In [ ]:
mark_pixel(ax_An, 3262) 

In [ ]:
mark_pixel(ax_An, 3183)
mark_pixel(ax_A, 3183)

In [ ]:
mark_pixel(ax_An, 2478)
mark_pixel(ax_A, 2478)

In [ ]:
mark_pixel(ax_An, 1028)
mark_pixel(ax_A, 1028)

In [ ]:
def mark_pixel(pixel):
    target_pixel = pixel
    y_val = intensity.loc[target_pixel]
    plt.plot(target_pixel, y_val, "ro", markersize=7, label=f"pixel {target_pixel}")

    plt.annotate(
        f"Pixel: {target_pixel}",
        xy=(target_pixel, y_val),
    )


In [ ]:
mark_pixel(3262)
mark_pixel(3346)
mark_pixel(3611)

In [ ]:
def plot_normalized_pixels_vs_power(df, pixel=3262, power_col="power" ):
    df_sorted = df.sort_index()
    #plt.figure(figsize=(10, 6), dpi=100)
    x_power = df_sorted[power_col]
    intensity = df_sorted[pixel]
    max_val = intensity.max()
    norm_intensity = intensity / max_val if max_val != 0 else intensity
    # line_label = label if label is not None else f"Pixel {pixel}"
    line_label = f"Pixel {pixel}"

    plt.plot( x_power, 
             norm_intensity, 
             marker="o", 
             linewidth=1.5, 
             markersize=5, 
             label=line_label, )

    plt.xlabel("power [kw]")
    plt.ylabel("normalized intensity")
    plt.legend()


In [ ]:
%matplotlib QtAgg
fig_title = "relative_intensities"
plt.figure(fig_title, figsize=(10, 6), dpi=100)



In [ ]:
plot_normalized_pixels_vs_power(spectr_A, pixel=3262)
plot_normalized_pixels_vs_power(spectr_A, pixel=3346)
plot_normalized_pixels_vs_power(spectr_A, pixel=3611)